In [ ]:
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pypdf
!pip install google-generativeai
!pip install streamlit
!pip install pyngrok
!pip install -U langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Unit 4 _ AT.pdf to Unit 4 _ AT.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Unit 4 _ AT.pdf")
documents = loader.load()

/tmp/ipykernel_1677/4096065761.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print(len(chunks))

15


In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embedding_model)
vector_store.save_local("faiss_index")

In [ ]:
query = "What is XML "

docs = vector_store.similarity_search(query, k=3)

for d in docs:
    print(d.page_content)

Unit 4: XML and Ajax 
 
1. What is XML? 
XML (eXtensible Markup Language) is a markup language used to store, transport, and 
share data. 
It focuses on describing data, not displaying it. 
XML is self-descriptive, meaning the tags clearly explain the meaning of the data. 
 
2. Why XML is Needed? 
 To exchange data between different systems 
 Platform-independent data representation 
 Used in web services, configuration files, and databases 
 HTML displays data, while XML defines and stores data 
 
3. Features of XML 
 User-defined tags 
 Case-sensitive 
 Human-readable and machine-readable 
 Hierarchical (tree structure) 
 Platform and language independent 
 Supports validation (DTD / XSD) 
 
4. Basic Structure of XML 
<?xml version="1.0" encoding="UTF-8"?> 
<student> 
    <name>Rahul</name> 
    <rollno>101</rollno> 
    <branch>Computer Engineering</branch> 
</student> 
Parts of XML Document:
Types of XML Parsers: 
1. DOM Parser 
 Loads entire XML into memory 
 Allows re

In [ ]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
def ask_question(question):

    docs = vector_store.similarity_search(
        question,
        k=3
    )

    context = "\n".join(
        [doc.page_content for doc in docs]
    )

    history = "\n".join(chat_history)

    prompt = f"""
    Previous Conversation:
    {history}

    Context:
    {context}

    Question:
    {question}

    Answer using context.
    """

    response = model.generate_content(prompt)

    chat_history.append(
        f"User: {question}"
    )

    chat_history.append(
        f"Bot: {response.text}"
    )

    return response.text

In [ ]:
chat_history = []

question = input("Ask: ")

answer = ask_question(question)

print(answer)

Ask: what is xml
XML (eXtensible Markup Language) is a markup language used to store, transport, and share data. It focuses on describing data, not displaying it, and is self-descriptive, meaning its tags clearly explain the meaning of the data.


In [ ]:
%%writefile app.py

import streamlit as st
import google.generativeai as genai

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters  import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import tempfile

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# Gemini API Key
genai.configure(api_key="YOUR_API_KEY")

model = genai.GenerativeModel("gemini-2.5-flash")

chat_history = []

st.title(" RAG Based PDF Chatbot")

uploaded_files = st.file_uploader(
    "Upload PDF Files",
    type="pdf",
    accept_multiple_files=True
)
if uploaded_files:

    all_documents = []

    for uploaded_file in uploaded_files:

        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
            tmp.write(uploaded_file.read())
            pdf_path = tmp.name

        loader = PyPDFLoader(pdf_path)
        documents = loader.load()

        all_documents.extend(documents)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(all_documents)

    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vector_store = FAISS.from_documents(
        chunks,
        embedding_model
    )

    st.success(
        f"{len(uploaded_files)} PDF(s) loaded successfully!"
    )

    question = st.text_input("Ask a Question")

    if question:

        docs = vector_store.similarity_search(
            question,
            k=3
        )

        context = "\n".join(
            [doc.page_content for doc in docs]
        )

        history = "\n".join(
            st.session_state.chat_history
        )

        prompt = f"""
        Previous Conversation:
        {history}

        Context:
        {context}

        Question:
        {question}

        Answer only from the provided context.
        If the answer is not available, say:
        'The information is not available in the uploaded documents.'
        """

        response = model.generate_content(prompt)

        st.session_state.chat_history.append(
            f"User: {question}"
        )

        st.session_state.chat_history.append(
            f"Bot: {response.text}"
        )

        st.write(response.text)

Writing app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok


ngrok.set_auth_token("YOUR_AUTH_TOKEN")

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://breach-cascade-census.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!git config --global user.name "techjay18"
!git config --global user.email "jaydesai18825@gmail.com"

In [ ]:
!git init


Reinitialized existing Git repository in /content/.git/


In [ ]:
%%writefile requirements.txt
streamlit
google-generativeai
langchain
langchain-community
langchain-text-splitters
sentence-transformers
faiss-cpu
pypdf
transformers
torch
accelerate
huggingface-hub

Overwriting requirements.txt


In [ ]:
%%writefile .gitignore

__pycache__/
*.pyc
*.pyo
*.pyd

.env

logs.txt

sample_data/

faiss_index/

*.pdf

.ipynb_checkpoints/

Overwriting .gitignore


In [ ]:
!ls

app.py	requirements.txt  sample_data


In [ ]:
!git commit -m "Initial commit - RAG PDF Chatbot"

On branch main
nothing to commit, working tree clean


In [ ]:
!git rm -r --cached drive


fatal: pathspec 'drive' did not match any files


In [ ]:
!git add app.py
!git add requirements.txt
!git add .gitignore
!git add .config

In [ ]:
!ls -la

total 32
drwxr-xr-x 1 root root 4096 Jul 28 17:20 .
drwxr-xr-x 1 root root 4096 Jul 28 17:17 ..
-rw-r--r-- 1 root root 2396 Jul 28 17:19 app.py
drwxr-xr-x 4 root root 4096 Jun  4 13:32 .config
drwxr-xr-x 8 root root 4096 Jul 28 18:52 .git
-rw-r--r-- 1 root root  104 Jul 28 18:51 .gitignore
-rw-r--r-- 1 root root  169 Jul 28 18:50 requirements.txt
drwxr-xr-x 1 root root 4096 Jun  4 13:32 sample_data


In [ ]:
!git status

On branch main
nothing to commit, working tree clean


In [ ]:
!git commit -m "Initial commit - RAG PDF Chatbot"


On branch main
nothing to commit, working tree clean


In [ ]:
!git remote add origin https://github.com/techjay18/RAG-PDF-Chatbot.git

In [ ]:
!git branch -M main

In [ ]:
!git branch


* main


In [ ]:
!git remote set-url origin https://techjay18/RAG-PDF-Chatbot.git

In [ ]:
!git config --global pull.rebase false

In [ ]:
!git merge --abort

In [ ]:
!git pull origin main --allow-unrelated-histories --no-edit

From https://github.com/techjay18/RAG-PDF-Chatbot
 * branch            main       -> FETCH_HEAD
Merge made by the 'ort' strategy.
 LICENSE   | 21 +++++++++++++++++++++
 README.md |  2 ++
 2 files changed, 23 insertions(+)
 create mode 100644 LICENSE
 create mode 100644 README.md


In [ ]:
!git push -u origin main

Enumerating objects: 27, done.
Counting objects: 100% (27/27), done.
Delta compression using up to 2 threads
Compressing objects: 100% (19/19), done.
Writing objects: 100% (26/26), 9.03 KiB | 1.81 MiB/s, done.
Total 26 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), done.
To https://github.com/techjay18/RAG-PDF-Chatbot.git
   7c65975..01b5d0b  main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.
